# CIFAR-100 / ResNet-18: SGD vs SAM vs Input Gradient Regularization
**Small feasibility pilot, not the final 100-epoch / three-seed experiment.** Upload this file to Colab, select a **T4 GPU (free Colab)**, and run cells in order. No local clone or manual repository edits are needed. The isolated legacy environment is inherited from the repository's reproduction notebooks; setup can take several minutes and requires internet. It does not restart Colab or replace Colab's Python.

**Question:** can explicitly reducing input sensitivity match SAM's clean accuracy without achieving equally low measured weight-space sharpness? The notebook collects evidence; it does not automatically decide between causal explanations.

| Setting | This pilot | Future proposed experiment |
|---|---|---|
| Dataset | CIFAR-100; balanced 1,000 train / 5,000 validation; other train examples unused | 45,000 train / 5,000 validation |
| Model | CIFAR ResNet-18, 3×3 stride-1 stem, no max-pool, 100 classes | Same |
| Final training | 2 epochs, seed 42 | 100 epochs, 3 independent seeds |
| Lambda search | 0, 0.01, 0.1; 1 epoch each on validation | Broader grid at an adequate training horizon |
| Optimizer | SGD, ordinary momentum 0.9; LR 0.1 with cosine decay | Same |
| Batch / precision | 32 / FP32 (T4 memory profile) | 128 / FP32 |
| Shared regularization | L2 weight decay 0.0005 on all parameters; clipping at norm 5 | Must be predeclared |
| SAM | Global L2 radius 0.05, one batch of 32 | Same unless explicitly redesigned |
| Clean test | All 10,000 official test images, final epoch only | Same |
| Sensitivity / sharpness / PGD | Same balanced 100-image test subset for all methods | Larger subsets and stronger diagnostics |

This pilot intentionally changes dataset and training duration from notebook `02`. It reuses its JAX/Flax ResNet block, ordinary-momentum optimizer, augmentation, SAM update convention and clean-first-pass BatchNorm state. It adds exact double-backprop IGR, validation-only lambda selection, and controlled evaluation. It does **not** run the authors' original WRN train_step or claim a reproduction of SAM's reported paper numbers.

**T4 first:** keeps all 100 classes and the original 32×32 resolution. Fewer classes would change the classification task; a balanced smaller training subset reduces runtime. Batch 32 reduces activation/double-backprop memory, while the test probe subset reduces evaluation cost. These choices change BatchNorm and SAM's perturbation batch size relative to batch 128, so this is a feasibility pilot, not an equivalent batch-128 experiment. It has not yet been GPU-validated. If the exact-batch preflight fails or exceeds memory, stop and preserve the log; do not infer success from SGD/SAM alone.

**Runtime reduction after the first T4 measurement:** the earlier 5,000-image / five-epoch profile was skipped by its safety gate (219 min training estimate; 340 min planning total, not actual training). This version uses 10 examples per class and two final epochs, with one-epoch lambda trials. It is only a functional/cost smoke experiment: do not use its low-data, undertrained accuracy to accept or reject the research hypothesis. Previous failed/skipped attempts remain separate; rerun setup to create a fresh folder using the installed environment.


In [ ]:
USE_DRIVE = False  # True: save logs/checkpoints/results to Google Drive (authorization required).
RUN_EXPERIMENT = True  # False: setup + numerical preflight + timing only.
PILOT_EPOCHS = 2
TUNE_EPOCHS = 1
SEEDS = [42]  # One seed only in this small pilot; no uncertainty across seeds.
LAMBDA_GRID = [0.0, 0.01, 0.1]  # Includes zero as a negative control.
MAX_WORKER_MINUTES = 60  # Total experiment worker cap; not a Colab billing cap.


## 1. Isolated setup
Python 3.8, JAX 0.2.6, CUDA jaxlib 0.1.57+cuda110, Flax 0.2.2, CUDA 11.0, cuDNN 8.0.5 (including conda build suffix), and CUDA 11.x ptxas. Reuses the hardened setup in notebook `05` and pins the imported repository source to an existing commit. `requirements.txt` is unchanged; only the previously documented runtime absl-py compatibility adjustment is used. No PyTorch migration.

Setup checks package versions, GPU initialization and a tiny JIT. This verifies the environment, not the full IGR experiment. Training preflight exercises actual batch-32 second-order IGR before any lambda trials. GPU memory snapshots are recorded around that test; they are not continuous peak-memory measurements. A setup/network failure stops execution with its log preserved. The new notebook has not yet been executed on a Colab GPU.


In [ ]:
import os, sys, json, time, hashlib, subprocess, shutil, tarfile, io, urllib.request
import threading, signal, uuid, csv, traceback, platform, fcntl
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path('/content/sam_igr_t4_pilot')
DRIVE = Path('/content/drive/MyDrive/sam_replication')
SOURCE_REVISION = '7f34662deec03341b78cccd4ce18ec2f7900429b'
REPO = ROOT / 'work' / 'sam'; PREFIX = ROOT / 'envs' / 'sam'; PYTHON = PREFIX / 'bin' / 'python'
SESSION = None

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False)); os.replace(str(tmp), str(path))

def command(args, log, environment=None, deadline=None, check=True):
    """Stream file logs; bound subprocesses even if the notebook kernel disappears."""
    args = list(map(str, args))
    # micromamba uses -c for channels; only Python's own -c contains code.
    python_inline = len(args) > 2 and Path(args[0]).name.startswith('python') and args[1] == '-c'
    if python_inline:
        compile(args[2], '<subprocess Python>', 'exec')
    log = Path(log); log.parent.mkdir(parents=True, exist_ok=True)
    remaining = int((deadline or (time.monotonic()+1800))-time.monotonic())
    if remaining <= 0: raise TimeoutError('Session deadline reached; no new process started')
    bounded = ['timeout', '--signal=INT', '--kill-after=15s', str(remaining)+'s']+args
    label = list(args[:5])
    if python_inline:
        label[2] = '<embedded Python>'
    print('Running:', ' '.join(label), flush=True)
    with log.open('a') as output:
        output.write('\nCOMMAND '+repr(args)+'\n'); output.flush()
        offset = log.stat().st_size
        process = subprocess.Popen(bounded, stdout=output, stderr=subprocess.STDOUT,
                                   env=environment, start_new_session=True)
        try:
            with log.open() as reader:
                reader.seek(offset); heartbeat = time.monotonic()
                while process.poll() is None:
                    chunk = reader.read()
                    if chunk: print(chunk, end='', flush=True)
                    if time.monotonic()-heartbeat > 30:
                        print('[still running; log:', log.name+']', flush=True); heartbeat = time.monotonic()
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
            if check and process.returncode: raise RuntimeError('Exit %s; see %s' % (process.returncode, log))
            return process.returncode
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try: process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL); process.wait()

def environment():
    env = os.environ.copy()
    env.update(PATH=str(PREFIX/'bin')+':'+env.get('PATH', ''),
        LD_LIBRARY_PATH=str(PREFIX/'lib')+':'+env.get('LD_LIBRARY_PATH', ''),
        PYTHONPATH=str(REPO.parent), CUDA_HOME=str(PREFIX), CUDA_PATH=str(PREFIX),
        CUDA_VISIBLE_DEVICES='0', XLA_PYTHON_CLIENT_PREALLOCATE='false', PYTHONHASHSEED='0',
        TFDS_DATA_DIR=str(ROOT/'tensorflow_datasets'), TF_DETERMINISTIC_OPS='0',
        TF_CUDNN_DETERMINISTIC='0',
        XLA_FLAGS='--xla_gpu_cuda_data_dir='+str(ROOT/'cuda')+
            ' --xla_gpu_deterministic_reductions='+'false')
    return env

VERSION_CHECK = '''
import sys
from importlib.metadata import version
assert sys.version_info[:2] == (3,8)
expected={'jax':'0.2.6','jaxlib':'0.1.57+cuda110','flax':'0.2.2','numpy':'1.18.5','tensorflow':'2.3.1'}
for package,want in expected.items():
    got=version(package); assert got==want,(package,got,want)
import numpy, tensorflow, flax, jax
print('Verified pinned packages:',expected,flush=True)
'''
GPU_CHECK = VERSION_CHECK + '''
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
assert len(jax.devices())==1 and jax.devices()[0].platform=='gpu'
x=jax.jit(lambda a:a@a)(jax.numpy.ones((2,2))); x.block_until_ready()
print('GPU initialization PASS:',jax.devices(),flush=True)
'''

def pinned_cuda_ready():
    packages = {}
    for path in (PREFIX/'conda-meta').glob('*.json'):
        record = json.loads(path.read_text())
        packages[record['name']] = record['version']
    return (packages.get('cudatoolkit', '').startswith('11.0.') and
            packages.get('cudnn', '').split('.')[:3] == ['8', '0', '5'] and packages.get('cuda-nvcc') == '11.3.122')

def setup(deadline):
    if not REPO.exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        command(['git', 'clone', '--branch', 'repro', 'https://github.com/Siddharth-magesh/sam-replication.git', REPO], SESSION/'setup.log', deadline=deadline)
        command(['git', '-C', REPO, 'checkout', '--detach', SOURCE_REVISION], SESSION/'setup.log', deadline=deadline)
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
    assert actual == SOURCE_REVISION, 'Existing runtime checkout differs; use a fresh ROOT.'
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True, timeout=30).strip(), 'Tracked source modifications'
    ready = False
    if PYTHON.exists() and (PREFIX/'bin'/'ptxas').exists() and pinned_cuda_ready():
        ready = command([PYTHON, '-c', VERSION_CHECK], SESSION/'environment_check.log', environment(), deadline, check=False) == 0
    if not ready:
        manager = ROOT/'bin'/'micromamba'
        if not manager.exists():
            with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest', timeout=120) as response:
                data = response.read()
            with tarfile.open(fileobj=io.BytesIO(data), mode='r:bz2') as archive:
                member = archive.getmember('bin/micromamba'); assert member.isfile()
                manager.parent.mkdir(parents=True, exist_ok=True)
                manager.write_bytes(archive.extractfile(member).read())
            manager.chmod(0o755)
        os.environ['MAMBA_ROOT_PREFIX'] = str(ROOT/'mamba')
        action = 'install' if PYTHON.exists() else 'create'
        command([manager, action, '-p', PREFIX, '-c', 'conda-forge', '--override-channels',
                 'python=3.8', 'pip=24.0', 'cudatoolkit=11.0', 'cudnn=8.0.5', '-y'], SESSION/'setup.log', deadline=deadline)
        command([manager, 'install', '-p', PREFIX, '-c', 'nvidia/label/cuda-11.3.1', '--override-channels',
                 'cuda-nvcc=11.3.122', '-y'], SESSION/'setup.log', deadline=deadline)
        lines = (REPO/'requirements.txt').read_text().splitlines()
        requirements = ROOT/'requirements_runtime.txt'
        requirements.write_text('\n'.join(s.replace('absl-py==0.11.0', 'absl-py==0.10.0')
            for s in lines if not s.startswith('jaxlib'))+'\n')
        command([PYTHON, '-m', 'pip', 'install', 'jaxlib==0.1.57+cuda110', '-f',
                 'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'], SESSION/'setup.log', deadline=deadline)
        command([PYTHON, '-m', 'pip', 'install', '-r', requirements], SESSION/'setup.log', deadline=deadline)
    assert pinned_cuda_ready(), 'CUDA/cuDNN pins differ; inspect setup log instead of upgrading JAX'
    atomic_json(SESSION/'cuda_packages.json', [json.loads(p.read_text()) for p in (PREFIX/'conda-meta').glob('*.json')
        if p.name.startswith(('cudatoolkit-', 'cudnn-', 'cuda-nvcc-'))])
    candidates = sorted(PREFIX.rglob('libdevice*.bc')); assert candidates, 'CUDA libdevice missing'
    libdir = ROOT/'cuda'/'nvvm'/'libdevice'; libdir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(candidates[0]), str(libdir/'libdevice.10.bc'))
    command([PREFIX/'bin'/'ptxas', '--version'], SESSION/'environment_check.log', environment(), deadline)
    command([PYTHON, '-c', GPU_CHECK], SESSION/'environment_check.log', environment(), deadline)
    command([PYTHON, '-m', 'pip', 'freeze'], SESSION/'package_versions.txt', environment(), deadline)


assert sys.platform == 'linux' and platform.machine() == 'x86_64', 'Select a Colab NVIDIA GPU runtime.'
GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip()
print('Allocated GPU:', GPU)
if 'T4' not in GPU:
    print('This notebook uses the T4 profile on the allocated GPU; timings are specific to that device.')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/sam_replication/igr_pilot')
else:
    OUTPUT_ROOT = ROOT/'runs'
SESSION = OUTPUT_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
SESSION.mkdir(parents=True, exist_ok=False)
print('Save this result folder:', SESSION)
setup_started = time.monotonic()
try:
    setup(time.monotonic()+1800)
finally:
    atomic_json(SESSION/'setup_timing.json', {'seconds': time.monotonic()-setup_started})
ENV = environment()
print('Setup passed. No training yet.')


## 2. Protocol and implementation
For raw pixels $x\in[0,1]^{32\times32\times3}$ the penalty is
$$P(w)=\frac1B\sum_i\|\nabla_{x_i}\ell(f_w^{\mathrm{eval}}(x_i),y_i)\|_2^2,\qquad L=\mathrm{CE}_{\mathrm{train}}+\tfrac{\mathrm{wd}}2\|w\|_2^2+\lambda P(w).$$
Normalization is inside the model. **BN running statistics are frozen in the penalty branch** using the state at the start of the step, ensuring examples do not interact through BN. The clean CE branch trains with ordinary batch statistics and updates running state once. This hybrid BN convention is a pilot design choice: it regularizes the inference function, not the batch-coupled training loss. Do not conflate the two objectives.

Gradients of **summed** per-example CE give unscaled individual input gradients under frozen BN; the penalty then averages squared norms across images. It is not the squared gradient of mean batch loss. Nested differentiation flows through input gradients into weights. No detached or finite-difference approximation is used. SAM includes L2 in both gradients, updates original weights, clips only the final update gradient, and retains first-pass BN, following notebook `02`.

All final methods start from the same seeded initialization and see the same per-epoch shuffle/augmentation. Lambda trials also start fresh with the same tuning seed and data. Selection uses **final tuning-epoch validation accuracy only**, ties favor the smallest lambda. The selected lambda is frozen before test scoring, then all methods retrain from scratch for two epochs. The single-epoch tuning horizon may select a poor lambda. If lambda=0 wins, label the IGR comparison inconclusive. Do not enlarge the grid after inspecting test results.

**Preflight:** analytic double-backprop check; invariance to duplicating a batch; actual-model frozen-BN per-example gradient consistency; finite updates for each method; lambda=0 versus SGD; nonzero-lambda influence on actual updates. Timings include synchronized input prep/transfers after one warm-up step. These are smoke/numerical diagnostics, not proof of full equivalence to the paper.


In [ ]:
WORKER_SOURCE = r'''

# Derived from notebook 02's JAX/Flax CIFAR ResNet pilot. See notebook protocol.
import os, sys, json, time, math, hashlib, platform, subprocess, traceback, gc
from pathlib import Path
import numpy as np
import tensorflow as tf
tf.config.experimental.set_visible_devices([], 'GPU')
import tensorflow_datasets as tfds
import jax
import jax.numpy as jnp
import flax
from flax import nn, optim, serialization
C = json.loads(Path(sys.argv[1]).read_text())
OUT = Path(C['output']); PHASE = sys.argv[2]
sys.path.insert(0, C['repo'])
from sam_jax.imagenet_models.resnet import ResNetBlock
B = C['batch_size']; METHODS = ('SGD','SAM','IGR')
BEGIN = time.monotonic()

def ready(tree):
    for v in jax.tree_leaves(tree):
        if hasattr(v, 'block_until_ready'): v.block_until_ready()
    return tree

def save(name, value):
    path=OUT/name; tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False)); tmp.replace(path)

def gpu_memory():
    try:
        raw=subprocess.check_output(['nvidia-smi','-i','0','--query-gpu=name,memory.used,memory.total',
            '--format=csv,noheader,nounits'],text=True,timeout=10).strip()
        name,used,total=raw.rsplit(',',2)
        return dict(name=name.strip(),used_mib=float(used),total_mib=float(total),
                    measurement='nvidia-smi snapshot, not true peak')
    except Exception as exc:
        return dict(error=str(exc))

def check_time():
    if time.monotonic()-BEGIN > C['worker_seconds']:
        raise TimeoutError('Worker time cap reached; partial runs cannot enter the final comparison.')

def finite(tree):
    assert all(np.isfinite(np.asarray(x)).all() for x in jax.tree_leaves(tree)), 'Non-finite value'

def norm2(tree):
    return sum(jnp.sum(x*x) for x in jax.tree_leaves(tree))

def l2(tree): return jnp.sqrt(norm2(tree))

def add(a,b): return jax.tree_multimap(lambda x,y:x+y,a,b)
def sub(a,b): return jax.tree_multimap(lambda x,y:x-y,a,b)
def mul(a,s): return jax.tree_map(lambda x:x*s,a)

assert jax.__version__=='0.2.6' and flax.__version__=='0.2.2'
assert len(jax.devices())==1 and jax.devices()[0].platform=='gpu', 'One NVIDIA GPU required'
# Seeds for splits, augmentation and initialization are explicit and separate.
builder=tfds.builder('cifar100:3.0.2', data_dir=C['data_dir'])
builder.download_and_prepare()
assert builder.info.splits['train'].num_examples==50000
train=tfds.as_numpy(builder.as_dataset(split='train',batch_size=-1))
images,labels=train['image'],train['label'].astype(np.int32)
rs=np.random.RandomState(C['split_seed']); ti=[]; vi=[]
for cls in range(100):
    ids=rs.permutation(np.flatnonzero(labels==cls))
    vi.extend(ids[:50]); ti.extend(ids[50:50+C['train_per_class']])
ti=np.array(ti,dtype=np.int32); vi=np.array(vi,dtype=np.int32)
assert len(set(ti.tolist()) & set(vi.tolist()))==0
assert len(ti)==C['train_per_class']*100 and len(vi)==5000
np.savez(OUT/'split_indices.npz',train=ti,validation=vi)
STEPS=len(ti)//B
assert STEPS>0

def batches(epoch, seed):
    rng=np.random.RandomState(seed+1000+epoch); order=rng.permutation(ti)
    for off in range(0,STEPS*B,B):
        ids=order[off:off+B]
        padded=np.pad(images[ids],((0,0),(4,4),(4,4),(0,0)))
        xy=rng.randint(0,9,(B,2)); flip=rng.rand(B)<0.5
        x=np.stack([padded[i,u:u+32,v:v+32] for i,(u,v) in enumerate(xy)])
        x[flip]=x[flip,:,::-1,:]
        yield x.astype(np.float32)/255.,labels[ids]

def eval_batches(xx, yy, ids):
    for off in range(0,len(ids),B):
        chunk=ids[off:off+B]; n=len(chunk)
        x=np.zeros((B,32,32,3),np.float32); y=np.zeros(B,np.int32)
        x[:n]=xx[chunk].astype(np.float32)/255.; y[:n]=yy[chunk]
        mask=(np.arange(B)<n).astype(np.float32)
        yield x,y,mask

class CifarResNet18(nn.Module):
    def apply(self,x,train=True):
        # Raw [0,1] pixel coordinates for IGR and attacks, normalization INSIDE model.
        x=(x-jnp.array([0.5071,0.4867,0.4408]))/jnp.array([0.2675,0.2565,0.2761])
        conv=nn.Conv.partial(bias=False,dtype=jnp.float32)
        bn=nn.BatchNorm.partial(use_running_average=not train,momentum=0.9,epsilon=1e-5,dtype=jnp.float32)
        x=nn.relu(bn(conv(x,64,(3,3),(1,1),padding='SAME',name='stem'),name='stem_bn'))
        for stage in range(4):
            for block in range(2):
                stride=(2,2) if stage>0 and block==0 else (1,1)
                x=ResNetBlock(x,64*2**stage,conv=conv,norm=bn,act=nn.relu,strides=stride,
                              name='stage%d_block%d'%(stage,block))
        return nn.Dense(jnp.mean(x,axis=(1,2)),100)

def fresh(seed):
    with nn.stateful() as state:
        _,params=CifarResNet18.init_by_shape(jax.random.PRNGKey(seed),[((B,32,32,3),jnp.float32)])
    model=nn.Model(CifarResNet18,params)
    return optim.Momentum(learning_rate=C['lr'],beta=C['momentum'],nesterov=False).create(model),state

def losses(logits,y):
    return -jnp.take_along_axis(jax.nn.log_softmax(logits),y[:,None],axis=1)[:,0]

def predict(model,state,x):
    with nn.stateful(state,mutable=False): return model(x,train=False)

def train_loss(model,state,x,y):
    with nn.stateful(state) as ns: z=model(x,train=True)
    ce=jnp.mean(losses(z,y))
    return ce+0.5*C['weight_decay']*norm2(model.params),(ns,ce)

# With frozen BN there is no cross-example dependence. Gradient of the SUM
# returns each image's CE gradient, without the 1/B scaling of gradient(mean CE).
def input_grads(model,state,x,y):
    return jax.grad(lambda pixels:jnp.sum(losses(predict(model,state,pixels),y)))(x)

def penalty(model,state,x,y):
    dx=input_grads(model,state,x,y)
    return jnp.mean(jnp.sum(dx*dx,axis=(1,2,3)))

def igr_loss(model,state,x,y,lam):
    base,(ns,ce)=train_loss(model,state,x,y)
    # Differentiate THROUGH dx to parameters. Only BN running state is constant.
    p=penalty(model,state,x,y)
    return base+lam*p,(ns,ce,p)

def make_step(method):
    @jax.jit
    def step(opt,state,x,y,lr,lam):
        if method=='IGR':
            (value,(ns,ce,p)),g=jax.value_and_grad(igr_loss,has_aux=True)(opt.target,state,x,y,lam)
        else:
            (value,(ns,ce)),g=jax.value_and_grad(train_loss,has_aux=True)(opt.target,state,x,y)
            p=jnp.float32(0.)
            if method=='SAM':
                delta=mul(g,C['rho']/jnp.maximum(l2(g),1e-12))
                perturbed=add(opt.target,delta)
                (_, _),g=jax.value_and_grad(train_loss,has_aux=True)(perturbed,state,x,y)
                # Update ORIGINAL weights; keep clean first-pass BN statistics only.
        gn=l2(g); g=mul(g,jnp.minimum(1.,C['clip']/jnp.maximum(gn,1e-12)))
        return opt.apply_gradient(g,learning_rate=lr),ns,dict(objective=value,clean_ce=ce,igr_penalty=p,gradient_norm=gn)
    return step
STEPS_FN={m:make_step(m) for m in METHODS}

@jax.jit
def clean_batch(model,state,x,y,mask):
    z=predict(model,state,x)
    return jnp.sum((jnp.argmax(z,axis=1)==y)*mask),jnp.sum(losses(z,y)*mask)

def evaluate(model,state,xx,yy,ids):
    acc=ce=0.
    for x,y,mask in eval_batches(xx,yy,ids):
        check_time(); a,b=ready(clean_batch(model,state,x,y,mask)); finite((a,b))
        acc+=float(a); ce+=float(b)
    return dict(accuracy=100*acc/len(ids),ce=ce/len(ids),n=len(ids))

@jax.jit
def sensitivity_batch(model,state,x,y,mask):
    dx=input_grads(model,state,x,y); sq=jnp.sum(dx*dx,axis=(1,2,3))
    return jnp.sum(jnp.sqrt(sq)*mask),jnp.sum(sq*mask)

@jax.jit
def pgd_batch(model,state,x,y,mask,key):
    eps=C['pgd_epsilon']; alpha=C['pgd_alpha']
    lo=jnp.maximum(0.,x-eps); hi=jnp.minimum(1.,x+eps)
    adv=jnp.clip(x+jax.random.uniform(key,x.shape,minval=-eps,maxval=eps),lo,hi)
    robust=(jnp.argmax(predict(model,state,x),axis=1)==y)
    def body(_,carry):
        adv,robust=carry
        robust=robust & (jnp.argmax(predict(model,state,adv),axis=1)==y)
        grad=input_grads(model,state,adv,y)
        adv=jnp.clip(adv+alpha*jnp.sign(grad),lo,hi)
        return adv,robust
    adv,robust=jax.lax.fori_loop(0,C['pgd_steps'],body,(adv,robust))
    robust=robust & (jnp.argmax(predict(model,state,adv),axis=1)==y)
    return jnp.sum(robust*mask),jnp.max(jnp.abs(adv-x)),jnp.min(adv),jnp.max(adv)

@jax.jit
def weight_batch(model,state,x,y,mask):
    def f(m): return jnp.sum(losses(predict(m,state,x),y)*mask)
    return jax.value_and_grad(f)(model)

@jax.jit
def weight_project(delta,radius):
    return mul(delta,jnp.minimum(1.,radius/jnp.maximum(l2(delta),1e-12)))

@jax.jit
def ascent(delta,g,radius):
    return weight_project(add(delta,mul(g,radius/2./jnp.maximum(l2(g),1e-12))),radius)

def probe_weight(model,state,xx,yy,ids):
    val=0.; grad=jax.tree_map(jnp.zeros_like,model)
    for x,y,mask in eval_batches(xx,yy,ids):
        check_time(); v,g=ready(weight_batch(model,state,x,y,mask)); finite((v,g))
        val+=float(v); grad=add(grad,g)
    return val/len(ids),mul(grad,1./len(ids))

def sharpness(model,state,xx,yy,ids,seed):
    base,_=probe_weight(model,state,xx,yy,ids); result=[]
    leaves,structure=jax.tree_flatten(model)
    for radius in C['sharp_radii']:
        best=base; best_norm=0.; trajectory=[]
        for restart in range(C['sharp_restarts']):
            keys=jax.random.split(jax.random.PRNGKey(seed+800+restart),len(leaves))
            delta=jax.tree_unflatten(structure,[jax.random.normal(k,p.shape,dtype=p.dtype) for k,p in zip(keys,leaves)])
            delta=mul(delta,radius/jnp.maximum(l2(delta),1e-12))
            for step in range(C['sharp_steps']+1):
                perturbed=add(model,delta)
                value,g=probe_weight(perturbed,state,xx,yy,ids)
                n=float(ready(l2(delta))); assert n <= radius+1e-5
                trajectory.append(dict(restart=restart,step=step,ce=value,l2=n))
                if value>best: best=value; best_norm=n
                if step<C['sharp_steps']: delta=ready(ascent(delta,g,radius))
        result.append(dict(radius=radius,base_ce=base,max_ce=best,loss_increase=best-base,
                           best_delta_l2=best_norm,weight_l2=float(ready(l2(model))),trajectory=trajectory))
    return result

def validate_igr_math():
    # Independent analytic mixed-derivative check, catches a detached penalty or
    # accidental gradient-of-batch-mean / batch-size scaling.
    x=jnp.array([[.2,.7],[.5,.1]],dtype=jnp.float32); w=jnp.array([.3,-.4],dtype=jnp.float32)
    def toy(ww,xx): return .5*jnp.square(xx@ww)
    def p(ww,xx):
        dx=jax.grad(lambda z:jnp.sum(toy(ww,z)))(xx)
        return jnp.mean(jnp.sum(dx*dx,axis=1))
    expected=jnp.mean(jnp.square(x@w))*jnp.sum(w*w)
    mixed=2*jnp.mean((x@w)[:,None]*x,axis=0)*jnp.sum(w*w)+2*w*jnp.mean(jnp.square(x@w))
    np.testing.assert_allclose(np.asarray(p(w,x)),np.asarray(expected),rtol=1e-5)
    np.testing.assert_allclose(np.asarray(jax.grad(p)(w,x)),np.asarray(mixed),rtol=1e-5,atol=1e-7)
    np.testing.assert_allclose(np.asarray(p(w,x)),np.asarray(p(w,jnp.concatenate([x,x]))),rtol=1e-5)
    assert float(ready(l2(jax.grad(p)(w,x))))>0
    return dict(toy_penalty=float(p(w,x)),mixed_gradient=np.asarray(mixed).tolist(),batch_duplication_invariant=True)

save(PHASE+'_environment.json',dict(python=platform.python_version(),jax=jax.__version__,flax=flax.__version__,
    numpy=np.__version__,tensorflow=tf.__version__,device=str(jax.devices()[0]),config=C,
    nvidia_smi=subprocess.check_output(['nvidia-smi'],text=True),steps_per_epoch=STEPS,
    train_examples=len(ti),validation_examples=len(vi),
    worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))

def preflight():
    diagnostics=validate_igr_math(); rows=[]
    save('preflight_progress.json',dict(status='starting',batch_size=B,gpu=gpu_memory()))
    opt,state=fresh(C['seeds'][0]); ready((opt,state))
    x,y=next(batches(0,C['seeds'][0]))
    diagnostics['parameter_count']=sum(p.size for p in jax.tree_leaves(opt.target.params))
    # Actual model independence from batch companions, BN frozen.
    dx=ready(jax.jit(input_grads)(opt.target,state,x[:2],y[:2]))
    single=ready(jax.jit(input_grads)(opt.target,state,x[:1],y[:1]))
    np.testing.assert_allclose(np.asarray(dx[:1]),np.asarray(single),rtol=2e-3,atol=1e-5)
    for method in METHODS:
        save('preflight_progress.json',dict(status='testing',method=method,batch_size=B,gpu=gpu_memory()))
        t=time.monotonic(); warmed=ready(STEPS_FN[method](opt,state,x,y,np.float32(C['lr']),np.float32(.1)))
        finite(warmed); compile_s=time.monotonic()-t
        change=float(ready(l2(sub(warmed[0].target,opt.target))))
        assert change>0, 'Parameters did not update'
        times=[]; candidate,ns=warmed[:2]
        it=batches(0,C['seeds'][0])
        for _ in range(C['timed_steps']):
            check_time(); t=time.monotonic(); xx,yy=next(it)
            candidate,ns,metrics=ready(STEPS_FN[method](candidate,ns,xx,yy,np.float32(C['lr']),np.float32(.1)))
            finite(metrics); times.append(time.monotonic()-t)
        row=dict(method=method,compile_warmup_seconds=compile_s,mean_step_seconds=float(np.mean(times)),
                 step_seconds=times,parameter_change_l2=change,
                 gpu_memory_after_step=gpu_memory(),
                 initial_diagnostics={k:float(v) for k,v in warmed[2].items()})
        rows.append(row); save('benchmark.json',rows); print(json.dumps(row),flush=True)
        del candidate,ns,warmed; gc.collect()
    # Zero-lambda update must match SGD (same initialization and clean BN update).
    a=ready(STEPS_FN['SGD'](opt,state,x,y,np.float32(.1),np.float32(0.)))
    b=ready(STEPS_FN['IGR'](opt,state,x,y,np.float32(.1),np.float32(0.)))
    rel=float(ready(l2(sub(a[0].target,b[0].target))/jnp.maximum(l2(a[0].target),1e-12)))
    assert rel<1e-4, ('IGR lambda=0 differs from SGD',rel)
    diagnostics['lambda_zero_parameter_relative_l2']=rel
    # Exact mixed gradient must affect an actual-model update, not just logged loss.
    d=ready(STEPS_FN['IGR'](opt,state,x,y,np.float32(.1),np.float32(.1)))
    effect=float(ready(l2(sub(d[0].target,b[0].target))))
    assert effect>0 and float(d[2]['igr_penalty'])>0
    diagnostics['nonzero_lambda_update_difference_l2']=effect
    diagnostics['status']='passed'; diagnostics['batch_size']=B
    diagnostics['gpu_memory_after_checks']=gpu_memory()
    save('preflight_progress.json',dict(status='passed',batch_size=B,gpu=gpu_memory()))
    save('preflight.json',diagnostics)
    print('PREFLIGHT PASSED',flush=True)

HISTORY=[]
def train_one(method,seed,epochs,lam,tag):
    opt,state=fresh(seed); ready((opt,state)); cost=0.
    for epoch in range(epochs):
        check_time(); t=time.monotonic(); ce_sum=pen_sum=0.
        for i,(x,y) in enumerate(batches(epoch,seed)):
            check_time(); fraction=(epoch*STEPS+i)/(epochs*STEPS)
            lr=np.float32(C['lr']*.5*(1+math.cos(math.pi*fraction)))
            opt,state,metrics=ready(STEPS_FN[method](opt,state,x,y,lr,np.float32(lam)))
            finite(metrics); ce_sum+=float(metrics['clean_ce']); pen_sum+=float(metrics['igr_penalty'])
        train_s=time.monotonic()-t; t=time.monotonic()
        val=evaluate(opt.target,state,images,labels,vi); val_s=time.monotonic()-t
        cost+=train_s+val_s
        row=dict(tag=tag,method=method,seed=seed,lam=lam,epoch=epoch+1,epochs=epochs,
                 val_accuracy=val['accuracy'],val_ce=val['ce'],train_clean_ce=ce_sum/STEPS,gpu_memory=gpu_memory(),
                 train_igr_penalty=pen_sum/STEPS,train_seconds=train_s,validation_seconds=val_s,
                 train_eval_seconds=cost,last_lr=float(lr))
        HISTORY.append(row); save('history.json',HISTORY); print(json.dumps(row),flush=True)
    return opt,state,val

def experiment():
    assert json.loads((OUT/'preflight.json').read_text())['status']=='passed'
    # Test examples are NOT loaded or scored during lambda selection.
    tuning=[]
    for lam in C['lambda_grid']:
        check_time(); opt,state,val=train_one('IGR',C['tune_seed'],C['tune_epochs'],lam,'tune_lambda_'+str(lam))
        tuning.append(dict(lam=lam,validation_accuracy=val['accuracy'],validation_ce=val['ce']))
        save('lambda_tuning.json',tuning); del opt,state; gc.collect()
    selected=sorted(tuning,key=lambda r:(-r['validation_accuracy'],r['lam']))[0]['lam']
    save('selection.json',dict(selected_lambda=selected,rule='Highest FINAL tuning-epoch validation accuracy; ties smallest lambda',
        test_used=False,tuning=tuning))
    print('Frozen lambda selection:',selected,flush=True)
    results=[]
    for seed in C['seeds']:
        for method in METHODS:
            lam=selected if method=='IGR' else 0.
            opt,state,val=train_one(method,seed,C['epochs'],lam,'final')
            # Store final-epoch parameters + BN; checkpoints permit later read-only evaluation, not training resume.
            checkpoint=OUT/('%s_seed%d.msgpack'%(method,seed))
            tmp=checkpoint.with_suffix('.tmp')
            tmp.write_bytes(serialization.to_bytes(dict(model=opt.target,state=state))); tmp.replace(checkpoint)
            test=tfds.as_numpy(builder.as_dataset(split='test',batch_size=-1))
            xx,yy=test['image'],test['label'].astype(np.int32)
            all_ids=np.arange(len(yy),dtype=np.int32)
            assert len(yy)==10000
            probe=[]; rng=np.random.RandomState(C['probe_seed'])
            for cls in range(100): probe.extend(rng.permutation(np.flatnonzero(yy==cls))[:C['probe_per_class']])
            probe=np.array(probe,dtype=np.int32); np.save(OUT/'test_probe_indices.npy',probe)
            clean=evaluate(opt.target,state,xx,yy,all_ids)
            clean_probe=evaluate(opt.target,state,xx,yy,probe)
            norm_sum=sq_sum=robust=0.; max_budget=0.
            for i,(x,y,mask) in enumerate(eval_batches(xx,yy,probe)):
                check_time()
                a,b=ready(sensitivity_batch(opt.target,state,x,y,mask)); finite((a,b))
                norm_sum+=float(a); sq_sum+=float(b)
                # Same attack random start per seed/batch for all methods.
                key=jax.random.PRNGKey(C['probe_seed']+seed+i)
                a,b,low,high=ready(pgd_batch(opt.target,state,x,y,mask,key)); finite((a,b,low,high))
                assert float(b)<=C['pgd_epsilon']+1e-6 and float(low)>=-1e-6 and float(high)<=1+1e-6
                robust+=float(a); max_budget=max(max_budget,float(b))
            sharp=sharpness(opt.target,state,xx,yy,probe,C['probe_seed']+seed)
            row=dict(method=method,seed=seed,lam=lam,epoch=C['epochs'],validation_accuracy=val['accuracy'],
                clean_test_accuracy=clean['accuracy'],clean_test_ce=clean['ce'],test_n=clean['n'],
                probe_clean_accuracy=clean_probe['accuracy'],probe_n=len(probe),input_gradient_l2=norm_sum/len(probe),
                input_gradient_squared_l2=sq_sum/len(probe),pgd_accuracy=100*robust/len(probe),
                pgd_max_observed_linf=max_budget,sharpness=sharp,weight_l2=float(ready(l2(opt.target.params))))
            assert row['pgd_accuracy']<=row['probe_clean_accuracy']+1e-6
            finite((row['input_gradient_l2'],row['input_gradient_squared_l2']))
            results.append(row); save('results.json',results)
            print('FINAL',json.dumps({k:v for k,v in row.items() if k!='sharpness'}),flush=True)
            del opt,state,test,xx,yy; gc.collect()
    return results

try:
    save(PHASE+'_status.json',dict(status='running'))
    if PHASE=='preflight': preflight()
    elif PHASE=='experiment': experiment()
    else: raise ValueError(PHASE)
    save(PHASE+'_status.json',dict(status='complete',wall_seconds=time.monotonic()-BEGIN,
                                  worker_hours=(time.monotonic()-BEGIN)/3600))
except BaseException as exc:
    save(PHASE+'_status.json',dict(status='failed',error=str(exc),traceback=traceback.format_exc(),
                                  wall_seconds=time.monotonic()-BEGIN,gpu_memory=gpu_memory(),
        hint='For T4 OOM, preserve this attempt. Reduce batch only in a new reviewed config; do not remove IGR or bypass preflight.'))
    raise


'''
compile(WORKER_SOURCE, '<igr-worker>', 'exec')
print('Embedded worker syntax checked.')


In [ ]:
CFG = dict(epochs=PILOT_EPOCHS, tune_epochs=TUNE_EPOCHS, seeds=SEEDS,
    tune_seed=42, split_seed=2026, probe_seed=567, train_per_class=10,
    batch_size=32, lr=0.1, momentum=0.9, weight_decay=0.0005, clip=5.0, rho=0.05,
    lambda_grid=LAMBDA_GRID, timed_steps=5, probe_per_class=1,
    pgd_epsilon=8/255, pgd_alpha=2/255, pgd_steps=10,
    sharp_radii=[0.05], sharp_steps=3, sharp_restarts=1,
    worker_seconds=MAX_WORKER_MINUTES*60, repo=str(REPO), output=str(SESSION),
    data_dir=str(ROOT/'tensorflow_datasets'), source_revision=SOURCE_REVISION,
    gpu=GPU, protocol='cifar100_igr_t4_smoke_v3')
assert PILOT_EPOCHS == 2 and TUNE_EPOCHS == 1 and SEEDS == [42], 'This is the fixed small pilot; prepare a reviewed full experiment separately.'
assert LAMBDA_GRID == [0.0,0.01,0.1] and 0 < MAX_WORKER_MINUTES <= 60
assert not (SESSION/'config.json').exists(), 'Rerun setup for a fresh folder; existing attempts are preserved.'
SCRIPT = SESSION/'worker.py'
SCRIPT.write_text(WORKER_SOURCE)
atomic_json(SESSION/'config.json', CFG)
MANIFEST = dict(worker_sha256=hashlib.sha256(SCRIPT.read_bytes()).hexdigest(),
                config_sha256=hashlib.sha256((SESSION/'config.json').read_bytes()).hexdigest())
atomic_json(SESSION/'manifest.json', MANIFEST)

def launch(phase):
    assert phase in ('preflight','experiment')
    assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest()==MANIFEST['worker_sha256']
    assert hashlib.sha256((SESSION/'config.json').read_bytes()).hexdigest()==MANIFEST['config_sha256']
    marker=SESSION/(phase+'_launch.json')
    assert not marker.exists(), 'Attempt already exists. Inspect it, or rerun setup/config for a fresh run.'
    if phase=='experiment':
        assert json.loads((SESSION/'preflight_status.json').read_text())['status']=='complete'
    atomic_json(marker,dict(status='running',manifest=MANIFEST))
    t=time.monotonic(); status='failed'
    try:
        command([PYTHON,'-u',SCRIPT,SESSION/'config.json',phase],SESSION/(phase+'.log'),ENV,
                deadline=time.monotonic()+CFG['worker_seconds'])
        assert json.loads((SESSION/(phase+'_status.json')).read_text())['status']=='complete'
        status='complete'
    finally:
        atomic_json(marker,dict(status=status,manifest=MANIFEST,wall_seconds=time.monotonic()-t))
        print('Attempt status:', status, '| log:', SESSION/(phase+'.log'))

launch('preflight')


## 3. Timing gate and pilot training
The five-step timings are estimates, not a promised runtime. The preflight has already run real batch-32 updates for all three methods and saved GPU memory snapshots. IGR requires mixed second derivatives and may be slower than SAM. The projection below includes the extra lambda-search runs and a 50% training allowance, plus a fixed evaluation allowance. Setup/download and future compilation may dominate. A hard subprocess deadline enforces the worker cap even if output is silent. If the projection exceeds the cap, training is skipped; the notebook never silently reduces epochs/data or changes methods.

This is equal final epochs, **not equal total compute**: IGR receives extra validation-search compute, which is reported separately. Final accuracy always comes from the last epoch, not the best observed epoch. Interrupted methods are excluded from final comparisons. Final model/BN checkpoints support later evaluation; optimizer/RNG resume is not implemented. Re-running creates a new experiment, not a continuation.


In [ ]:
benchmark=json.loads((SESSION/'benchmark.json').read_text())
means={r['method']:r['mean_step_seconds'] for r in benchmark}
print('Measured seconds per step:', {m:round(v,3) for m,v in means.items()})
steps=(CFG['train_per_class']*100)//CFG['batch_size']
training_estimate=steps*(CFG['epochs']*len(CFG['seeds'])*sum(means.values())+
    CFG['tune_epochs']*len(CFG['lambda_grid'])*means['IGR'])
warmup_estimate=sum(r['compile_warmup_seconds'] for r in benchmark)
projection=1.5*training_estimate+warmup_estimate+600  # Evaluation allowance is unmeasured.
atomic_json(SESSION/'planning_estimate.json',dict(training_seconds=training_estimate,
    conservative_total_seconds=projection,evaluation_allowance_seconds=600,measured_runtime=False))
print('Estimated training only:',round(training_estimate/60,1),'min')
print('Planning total with allowances:',round(projection/60,1),'min; cap:',MAX_WORKER_MINUTES,'min')
if not RUN_EXPERIMENT:
    atomic_json(SESSION/'pilot_decision.json',dict(status='disabled',training_started=False))
    print('Training disabled. No lambda selection or final test results.')
elif projection > CFG['worker_seconds']:
    atomic_json(SESSION/'pilot_decision.json',dict(status='skipped_budget',training_started=False,
        projected_seconds=projection,cap_seconds=CFG['worker_seconds']))
    print('SKIPPED: estimate exceeds pilot cap. Training did not start; no final accuracy metrics are expected.')
else:
    atomic_json(SESSION/'pilot_decision.json',dict(status='started',training_started=True,
        projected_seconds=projection,cap_seconds=CFG['worker_seconds']))
    launch('experiment')


## 4. Metrics, results and interpretation
* **Clean test accuracy:** all 10,000 official test images; report percentage points versus SGD.
* **Input sensitivity:** mean per-image $\|\nabla_x\ell\|_2$ and mean squared L2 on the same balanced 100-image test subset. Lower is less local CE sensitivity, not a robustness certificate.
* **Weight sharpness:** maximum observed increase in mean **unregularized inference CE** on that subset under global absolute L2 weight perturbations, radius 0.05. BN running statistics are fixed; all trainable parameters, including BN affine terms, are perturbed. Three projected gradient-ascent updates, step radius/2, one seeded random sphere start per radius, and the unperturbed candidate. This is an **optimization lower bound / proxy**, not the exact worst-case loss or Hessian flatness. It is parameterization-sensitive; weight L2 is reported for context. Low loss or sharpness does not establish a causal mechanism.
* **PGD-10 accuracy:** untargeted CE, raw-pixel L∞ epsilon 8/255, step 2/255, one shared seeded random start; clip to both image and attack bounds; frozen BN. An example is robust only if correct on clean input **and every visited iterate**. Compare with clean accuracy on the **same subset**, not full-test accuracy. This short attack can overestimate robustness; the elaborate study needs stronger/multiple-restart attacks and masking checks.

No confidence intervals or significance claims with one seed. “Comparable accuracy” and “significantly higher sharpness” cannot be established by this pilot. If IGR lowers sensitivity but has poor accuracy, inspect undertraining, penalty scale and lambda-selection adequacy before rejecting the hypothesis. If SAM shows no clean gain over SGD, there is no demonstrated SAM gain for IGR to match. Neither outcome settles causality. The future experiment needs three independent seeds, a predeclared accuracy-equivalence margin and uncertainty estimates, adequate tuning, stronger sharpness/attack optimization and compute accounting.

Sources: [SAM](https://arxiv.org/abs/2010.01412), [input-gradient regularization](https://arxiv.org/abs/1711.09404), [PGD adversarial training](https://arxiv.org/abs/1706.06083), [JAX higher-order differentiation](https://docs.jax.dev/en/latest/advanced_autodiff.html). The notebook's BN convention and sharpness protocol are explicitly defined above rather than claimed to replicate those papers.

Only one example per class enters the 100-image probe. This keeps the free-T4 run small but makes robustness/sharpness estimates noisy; use these to check that measurement works. Full clean test accuracy still uses all 10,000 test images.


In [ ]:
# Reporting runs in the pinned interpreter, avoiding changes to Colab's libraries.
REPORT_SOURCE = r'''
import sys,json,csv
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
p=Path(sys.argv[1])
if (p/'history.json').exists():
    h=pd.DataFrame(json.loads((p/'history.json').read_text())); h.to_csv(p/'history.csv',index=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for m in ('SGD','SAM','IGR'):
        a=h[(h.tag=='final') & (h.method==m)]
        if a.empty: continue
        axes[0].plot(a.epoch,a.val_accuracy,'o-',label=m)
        axes[1].plot(a.train_eval_seconds/60,a.val_accuracy,'o-',label=m)
        axes[2].plot(a.epoch,a.train_clean_ce,'o-',label=m)
    for ax in axes:
        if ax.lines: ax.legend()
        ax.grid(alpha=.2)
    axes[0].set(xlabel='Epoch',ylabel='Validation accuracy (%)',title='Final runs only')
    axes[1].set(xlabel='Training + validation minutes',ylabel='Validation accuracy (%)',title='Includes JIT; excludes tuning/test probes')
    axes[2].set(xlabel='Epoch',ylabel='Training clean CE',title='Clean first-pass CE, no penalty/L2')
    fig.tight_layout(); fig.savefig(p/'learning_curves.png',dpi=160); plt.close(fig)
    cost=h.groupby(['tag','method'])[['train_seconds','validation_seconds']].sum()
    cost.to_csv(p/'training_compute.csv'); print('Measured training/validation costs (seconds):'); print(cost)
if (p/'selection.json').exists(): print('Lambda selection:',json.loads((p/'selection.json').read_text()))
if not (p/'results.json').exists():
    decision=p/'pilot_decision.json'
    status=json.loads(decision.read_text()).get('status') if decision.exists() else 'unknown'
    if status=='skipped_budget':
        print('No final metrics: the timing gate skipped training because its estimate exceeded the cap. This is not a reporting error.')
    elif status=='disabled':
        print('No final metrics: training was disabled.')
    else:
        print('No final method metrics yet. Inspect experiment status/log files.')
    sys.exit(0)
raw=json.loads((p/'results.json').read_text()); rows=[]
for r in raw:
    row={k:v for k,v in r.items() if k!='sharpness'}
    for s in r['sharpness']: row['sharpness_rho_'+str(s['radius'])]=s['loss_increase']
    rows.append(row)
df=pd.DataFrame(rows)
for seed in df.seed.unique():
    base=df[(df.seed==seed)&(df.method=='SGD')]
    if not base.empty: df.loc[df.seed==seed,'accuracy_delta_vs_sgd_pp']=df.loc[df.seed==seed,'clean_test_accuracy']-float(base.iloc[0].clean_test_accuracy)
df.to_csv(p/'final_metrics.csv',index=False)
complete=len(df)==3 and set(df.method)=={'SGD','SAM','IGR'}
print('COMPLETE PILOT' if complete else 'INCOMPLETE: no full three-method comparison')
cols=['method','clean_test_accuracy','accuracy_delta_vs_sgd_pp','input_gradient_l2','input_gradient_squared_l2','sharpness_rho_0.05','probe_clean_accuracy','pgd_accuracy']
print(df[[c for c in cols if c in df]].to_string(index=False))
fig,axes=plt.subplots(1,4,figsize=(17,4))
for ax,col,title in zip(axes,['clean_test_accuracy','input_gradient_l2','sharpness_rho_0.05','pgd_accuracy'],['Full test accuracy (%)','Mean input-gradient L2','Sharpness proxy, rho=0.05','PGD-10 subset accuracy (%)']):
    ax.bar(df.method,df[col],color=['#4277b6','#e58b2a','#399c87'][:len(df)])
    ax.set_title(title); ax.grid(axis='y',alpha=.2); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(p/'final_metrics.png',dpi=160)
print('One seed and two epochs on 1,000 training images. Functional/cost smoke only; no statistical/causal conclusion.')
'''
compile(REPORT_SOURCE,'<report>','exec')
report=SESSION/'report.py'; report.write_text(REPORT_SOURCE)
command([PYTHON,report,SESSION],SESSION/'report.log',ENV,deadline=time.monotonic()+180)
from IPython.display import display, Image
for name in ['learning_curves.png','final_metrics.png']:
    if (SESSION/name).exists(): display(Image(filename=str(SESSION/name)))


## 5. Download and preserve results
The ZIP contains configuration, source hashes, exact worker/report source, split/probe indices, lambda search, complete and failed status/logs, measured timing, metrics and plots. Final model checkpoints are deliberately excluded from the default ZIP because they are large; they remain in the printed result folder. Set `USE_DRIVE=True` **before setup** if you want those checkpoints to survive a Colab reset. Otherwise download/copy anything you need before ending the session.

This notebook does not disconnect Colab or stop billing. Its cap controls worker processes only. If a training cell raises an error, you can still run the reporting and download cells to preserve partial evidence. Full GPU runtime compatibility, speed and experiment outcomes remain to be verified by execution.


In [ ]:
import zipfile
archive = ROOT/(SESSION.name+'_igr_pilot.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in SESSION.rglob('*'):
        if file.is_file() and file.suffix not in ('.msgpack','.tmp'):
            z.write(file,str(Path(SESSION.name)/file.relative_to(SESSION)))
print('Saved:',archive)
from google.colab import files
files.download(str(archive))
